In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, HBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# IDEAL AND BANDLIMITED WHITE NOISE
# ============================================================

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="font-family:Arial, sans-serif; font-size:16px; line-height:1.40; width:1050px;">

<div style="font-size:22px; font-weight:bold; color:#173f8a; margin-bottom:8px;">
Ideal and Bandlimited White Noise
</div>

<div style="margin-bottom:4px;">
Ideal white noise has a constant PSD over an infinite frequency range and an impulse autocorrelation.
</div>

<div style="margin-bottom:4px;">
Bandlimited white noise has a constant PSD only for |f| ≤ W and therefore finite total power.
</div>

<div style="margin-bottom:4px;">
Its autocorrelation is a sinc function, Rₓₓ(τ) = 2WS₀ sinc(2Wτ).
</div>

<div>
<b>This notebook:</b> shows how increasing the noise bandwidth makes the autocorrelation increasingly narrow.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='170px')

W_slider = FloatSlider(min=2.0, max=20.0, step=1.0, value=8.0, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)
S0_slider = FloatSlider(min=0.25, max=2.0, step=0.25, value=1.0, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

# ============================================================
# VALUE LABELS
# ============================================================

W_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">8.0</div>')
S0_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>')

def update_W_value(change):
    W_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{W_slider.value:.1f}</div>'

def update_S0_value(change):
    S0_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{S0_slider.value:.2f}</div>'

W_slider.observe(update_W_value, names='value')
S0_slider.observe(update_S0_value, names='value')

# ============================================================
# LABELS
# ============================================================

W_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Bandwidth W:</div>')
S0_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">PSD level S₀:</div>')

# ============================================================
# CONTROLS
# ============================================================

controls_grid = GridBox(children=[W_label, W_slider, W_value, S0_label, S0_slider, S0_value], layout=Layout(width='720px', grid_template_columns='115px 170px 60px 115px 170px 60px', grid_template_rows='34px', grid_gap='5px 8px', align_items='center', overflow='hidden'))

controls_card = VBox([HTML('<div style="font-family:Arial; font-size:17px; font-weight:bold; color:#173f8a; margin-bottom:5px;">Parameters</div>'), controls_grid], layout=Layout(width='750px', padding='10px 14px', border='1px solid #b9cae7', margin='10px 0px 8px 0px', overflow='hidden'))

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# MAIN FUNCTION
# ============================================================

def plot_white_noise(W=8.0, S0=1.0):

    f = np.linspace(-25.0, 25.0, 2500)
    tau = np.linspace(-1.0, 1.0, 2500)

    S_band = np.where(np.abs(f) <= W, S0, 0.0)

    R_band = 2.0 * W * S0 * np.sinc(2.0 * W * tau)

    R_normalized = np.sinc(2.0 * W * tau)

    fig = plt.figure(figsize=(10.4, 6.5))

    gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.30)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, :])

    # --------------------------------------------------------
    # IDEAL WHITE-NOISE PSD
    # --------------------------------------------------------

    ax1.plot(f, S0 * np.ones_like(f), linewidth=2.0)

    ax1.set_xlim(-25, 25)
    ax1.set_ylim(0, 2.2)
    ax1.set_xlabel('Frequency f', fontsize=11)
    ax1.set_ylabel('PSD', fontsize=11)
    ax1.set_title('Ideal White Noise: Conceptual PSD', fontsize=13, pad=9)
    ax1.grid(True, linestyle=':', alpha=0.5)

    # --------------------------------------------------------
    # BANDLIMITED PSD
    # --------------------------------------------------------

    ax2.plot(f, S_band, linewidth=2.0)

    ax2.axvline(-W, linestyle='--', linewidth=1.0)
    ax2.axvline(W, linestyle='--', linewidth=1.0)

    ax2.set_xlim(-25, 25)
    ax2.set_ylim(0, 2.2)
    ax2.set_xlabel('Frequency f', fontsize=11)
    ax2.set_ylabel('PSD', fontsize=11)
    ax2.set_title('Bandlimited White Noise PSD', fontsize=13, pad=9)
    ax2.grid(True, linestyle=':', alpha=0.5)

    # --------------------------------------------------------
    # AUTOCORRELATION
    # --------------------------------------------------------

    ax3.plot(tau, R_normalized, linewidth=2.0)

    ax3.axhline(0, linewidth=0.8)
    ax3.axvline(0, linewidth=0.8, linestyle=':')

    zero_spacing = 1.0 / (2.0 * W)

    for k in range(-5, 6):
        if k != 0:
            location = k * zero_spacing
            if -1.0 <= location <= 1.0:
                ax3.axvline(location, linestyle=':', linewidth=0.7, alpha=0.5)

    ax3.set_xlim(-1.0, 1.0)
    ax3.set_ylim(-0.30, 1.10)
    ax3.set_xlabel('Lag τ', fontsize=11)
    ax3.set_ylabel('Normalized Rₓₓ(τ)', fontsize=11)
    ax3.set_title('Autocorrelation of Bandlimited White Noise', fontsize=13, pad=9)
    ax3.grid(True, linestyle=':', alpha=0.5)

    fig.subplots_adjust(left=0.08, right=0.97, top=0.93, bottom=0.10)

    plt.show()
    plt.close(fig)

    result_html.value = f"""
    <div style="font-family:Arial; font-size:15px; line-height:1.42; width:900px; padding:10px 14px; border:1px solid #d7c38d; background:#fffbed; box-sizing:border-box;">
    <b>Total noise power:</b> Rₓₓ(0) = 2WS₀ = {2.0 * W * S0:.4f}
    &nbsp;&nbsp;&nbsp;
    <b>First autocorrelation zero:</b> τ = ±1/(2W) = ±{1.0 / (2.0 * W):.5f}
    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(plot_white_noise, {'W': W_slider, 'S0': S0_slider})

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="font-family:Arial; font-size:15px; line-height:1.42; width:1050px; padding:11px 15px; border:1px solid #c8dfce; background:#f8fcf9; box-sizing:border-box; margin-top:6px;">
<div style="font-size:18px; font-weight:bold; color:#197b35; margin-bottom:6px;">Interpretation of the Results</div>
<div style="margin-bottom:4px;">A finite rectangular spectrum produces a sinc autocorrelation rather than an ideal impulse.</div>
<div style="margin-bottom:4px;">Increasing W makes the main lobe narrower and moves the autocorrelation zeros toward τ = 0.</div>
<div>In the limiting idealization W → ∞, the autocorrelation approaches an impulse.</div>
</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox([documentation, controls_card, output, result_html, interpretation], layout=Layout(width='1050px', overflow='hidden'))

display(main_layout)